In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = pow(2,40).__str__()
import cv2

import torch

DATA_PATH = "E:/ML/UBC"
print(DATA_PATH)


E:/ML/UBC


In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


## Dataset

In [3]:
testList = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))
testList.head()

,image_id,image_width,image_height
0,41,28469,16987


In [5]:
from sklearn.preprocessing import LabelEncoder
import warnings
import pickle 

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open(os.path.join("./", 'LabelEncoder5.pkl'), 'rb') as f:
        enc = pickle.load(f) 
    print(enc.classes_)
    print(enc.transform(["LGSC"]))

['CC' 'EC' 'HGSC' 'LGSC' 'MC']
[3]


In [16]:
IMG_SIZE = (384, 384)
eps=1e-12

In [15]:
PatchSize=512

#Testing Flag
DEV=True

if DEV:
    allIds = [41, 99, 13568,17637,12442]
else:
    allIds = testList["image_id"].unique()

def getPatches(data):
    patches=[]
#     if data.shape[0]<5000 and data.shape[1]<5000:
#         #probably TMA
#         PatchSize=1024
#     else:
#         #WSI
#         PatchSize=512
    NRows = data.shape[0]//PatchSize
    NCols = data.shape[1]//PatchSize
    div = np.max([(NRows*NCols)//1500, 1]).astype(np.int64)
#     print(f"{div=}")
    
    for j in range(NRows):
        yPos = PatchSize*j
        for i in range(NCols):
            # Only take every n-th patch due to memory restrictions
            if (j*NCols + i)%div == 0:
                xPos = PatchSize*i
                patch = data[yPos:yPos+PatchSize, xPos:xPos+PatchSize]
                hist , _ = np.histogram(patch, bins=10, range=(0.0,255.0))
                # If most pixels are very bright or dark the patch is outside the relevant area
                if ((np.sum(hist[2:-2]))/(np.sum(hist)+eps)) < 0.1 or (hist[0])/np.sum(hist)>0.3:
                    continue
                w, h = patch.shape[1], patch.shape[0]
                if not (w == IMG_SIZE[0] and h == IMG_SIZE[1]):
                    patch = cv2.resize(patch, IMG_SIZE)
                patches.append(patch)
#     patches=[]
    return np.array(patches).astype(np.float32)/255.0


def getData(id):
    if DEV:
        imFile =  os.path.join(DATA_PATH, "train_images_test", str(id)+".png")
    else:
        imFile =  os.path.join(DATA_PATH, "test_images", str(id)+".png")
    if os.path.exists(imFile):
#         data = cv2.imread(imFile)[:,:,::-1]
        data = cv2.imread(imFile)
        data = cv2.cvtColor(data, cv2.COLOR_BGR2RGB ) 
        patches = getPatches(data)
        del data
        return patches, id
    else:
        return np.array([]).astype(np.float32), id


def generator():
    for trainId in allIds:
        yield getData(trainId)


In [12]:
X, fid = getData(99)

In [7]:
# g = generator()
# testInstance = next(g)
# testInstance[0].shape

In [8]:
# # plt.imshow(testInstance[0][0,:,:,:])
# print("id ", testInstance[1])
# print("max ", np.max(testInstance[0]))
# print("len ", len(testInstance[0]))
# print("shape ", testInstance[0].shape)


# plt.figure(figsize=(12,12))
# for i in range(49):
#     plt.subplot(7,7,i+1)
#     plt.imshow(testInstance[0][i,:,:,:])
#     _= plt.axis("off")


In [9]:
# del testInstance, g

## Stage 1: Detection

In [23]:
from torch import nn
import torch
import timm


classifier = timm.create_model('resnet34d', pretrained=False, num_classes=len(enc.classes_))

checkpoint = torch.load(os.path.join("./", "seg_resnet34d_384_5_epoch_15_ValAcc_0.9726.pt"))
print(classifier.load_state_dict(checkpoint['model_state_dict']))

del checkpoint

detector = timm.create_model('resnet18', pretrained=False, num_classes=1)

checkpoint2 = torch.load(os.path.join("./", "B_resnet18_384_epoch_10_ValAcc_0.9635.pt"))
print(detector.load_state_dict(checkpoint2['model_state_dict']))

del checkpoint2

<All keys matched successfully>
<All keys matched successfully>


In [46]:
from torchvision.transforms import v2


detections={}
result=[]
threshold = 0.5

transforms = v2.Compose([
    v2.ToDtype(torch.float32),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Start Test...")

detector = detector.to(device)
detector.eval()

classifier = classifier.to(device)
classifier.eval()

for batch, fid in enumerate(allIds):
    X, _ = getData(fid)
    X = torch.Tensor(X)
    if X.shape[0] == 0:
        result.append([fid, "HGSC"])
    elif not X.shape[1] == IMG_SIZE[0] or not X.shape[2] == IMG_SIZE[1] or not X.shape[3]==3:
        result.append([fid, "HGSC"])
    else:
        X = X.movedim(-1,1)
    
        #Detection
        X = transforms(X)
        print(batch, fid, X.shape)
        detectionsTemp=[]
        totalNum = X.shape[0]
        if totalNum>20:
            splits = np.linspace(0,totalNum, totalNum//10, dtype=np.int32)
            for j,spl in enumerate(splits):
                if j==splits.shape[0]-1:
                    break
                splittedX = X[spl: splits[j+1],:]
                splittedX = splittedX.to(device)
                preds = detector(splittedX)
                preds = torch.nn.functional.sigmoid(preds)
                preds = torch.flatten(preds) > threshold
                detectionsTemp = [*detectionsTemp, *preds.detach().cpu().numpy()]
                del splittedX, preds
        else:
            X = X.to(device)
            preds = detector(X)
            preds = torch.nn.functional.sigmoid(preds)
            preds = torch.flatten(preds) > threshold
            detectionsTemp = [ *preds.detach().cpu().numpy()]
            del preds
            
        detectionsTemp = np.array(detectionsTemp)
#         detections[fid] = detectionsTemp
        
        cancerousPatches = X[detectionsTemp]
        
        del X
        
        print(cancerousPatches.shape)
        
        #Prediction
        predictions=[]
        totalNum = cancerousPatches.shape[0]
        if totalNum == 0:
            result.append([fid, "HGSC"])
            continue
        elif totalNum>20:
            splits = np.linspace(0,totalNum, totalNum//10, dtype=np.int32)
            for j,spl in enumerate(splits):
                if j==splits.shape[0]-1:
                    break
                splittedX = cancerousPatches[spl: splits[j+1],:]
                splittedX = splittedX.to(device)

                preds = classifier(splittedX)
                preds = torch.nn.functional.softmax(preds, dim=1)
                predictions = [*predictions, *preds.detach().cpu().numpy()]
                del splittedX, preds
        else:
            cancerousPatches = cancerousPatches.to(device)
            preds = classifier(cancerousPatches)
            preds = torch.nn.functional.softmax(preds, dim=1)
            predictions = [ *preds.detach().cpu().numpy()]
            del preds
            
        pred = np.mean(predictions, axis=0)
#         pred = np.max(predictions, axis=0)

#         print(pred)
        targetLabel = np.argmax(pred)
        prob = pred[targetLabel]

        label = enc.inverse_transform([targetLabel])[0]
        result.append([fid, label])
        del cancerousPatches
        torch.cuda.empty_cache()


Start Test...


  0%|          | 0/100 [00:00<?, ?it/s]

0 41 torch.Size([1, 3, 384, 384])


  3%|▎         | 3/100 [00:00<00:12,  7.83it/s]

torch.Size([0, 3, 384, 384])
1 99 torch.Size([1, 3, 384, 384])
torch.Size([0, 3, 384, 384])
2 13568 torch.Size([1, 3, 384, 384])
torch.Size([0, 3, 384, 384])
3 17637 torch.Size([1, 3, 384, 384])
torch.Size([0, 3, 384, 384])
4 12442 torch.Size([1, 3, 384, 384])
torch.Size([0, 3, 384, 384])
0 41 torch.Size([2, 3, 384, 384])
torch.Size([0, 3, 384, 384])
1 99 torch.Size([2, 3, 384, 384])
torch.Size([0, 3, 384, 384])
2 13568 torch.Size([2, 3, 384, 384])
torch.Size([0, 3, 384, 384])
3 17637 torch.Size([2, 3, 384, 384])
torch.Size([0, 3, 384, 384])
4 12442 torch.Size([2, 3, 384, 384])
torch.Size([0, 3, 384, 384])
0 41 torch.Size([3, 3, 384, 384])
torch.Size([0, 3, 384, 384])
1 99 torch.Size([3, 3, 384, 384])


...

torch.Size([0, 3, 384, 384])
0 41 torch.Size([98, 3, 384, 384])
torch.Size([0, 3, 384, 384])
1 99 torch.Size([98, 3, 384, 384])
torch.Size([0, 3, 384, 384])
2 13568 torch.Size([98, 3, 384, 384])
torch.Size([0, 3, 384, 384])
3 17637 torch.Size([98, 3, 384, 384])
torch.Size([0, 3, 384, 384])
4 12442 torch.Size([98, 3, 384, 384])


 99%|█████████▉| 99/100 [03:37<00:04,  4.20s/it]

torch.Size([0, 3, 384, 384])
0 41 torch.Size([99, 3, 384, 384])
torch.Size([0, 3, 384, 384])
1 99 torch.Size([99, 3, 384, 384])
torch.Size([0, 3, 384, 384])
2 13568 torch.Size([99, 3, 384, 384])
torch.Size([0, 3, 384, 384])
3 17637 torch.Size([99, 3, 384, 384])
torch.Size([0, 3, 384, 384])
4 12442 torch.Size([99, 3, 384, 384])


100%|██████████| 100/100 [03:42<00:00,  2.22s/it]

torch.Size([0, 3, 384, 384])


In [38]:
detectionsTemp

array([False])

## Stage 2: Classification

In [12]:

# classifier = timm.create_model('resnet34d', pretrained=False, num_classes=len(enc.classes_))

# checkpoint = torch.load(os.path.join("/kaggle/input/ubc-model/", "seg_resnet34d_384_5_epoch_15_ValAcc_0.9726.pt"))
# print(classifier.load_state_dict(checkpoint['model_state_dict']))

# del checkpoint

In [13]:
# from torchvision.transforms import v2

# result=[]

# threshold = 0.5

# transforms = v2.Compose([
#     v2.ToDtype(torch.float32),
#     v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

# print("Start Test...")


# classifier = classifier.to(device)
# classifier.eval()

# for batch, fid in enumerate(allIds):
#     X, _ = getData(fid)
#     X = torch.Tensor(X)
#     if X.shape[0] == 0:
#         result.append([fid, "Other"])
#     elif not X.shape[1] == IMG_SIZE[0] or not X.shape[2] == IMG_SIZE[1] or not X.shape[3]==3:
#         result.append([fid, "HGSC"])
#     else:
#         X = X.movedim(-1,1)
            
#         detectionsTemp = detections[fid]
#         cancerousPatches = X[detectionsTemp]
        
#         del X
        
#         print(cancerousPatches.shape)
        
#         #Prediction
#         predictions=[]
#         totalNum = cancerousPatches.shape[0]
#         if totalNum == 0:
#             result.append([fid, "Other"])
#         elif totalNum>10:
#             splits = np.linspace(0,totalNum, totalNum//10, dtype=np.int32)
#             for j,spl in enumerate(splits):
#                 if j==splits.shape[0]-1:
#                     break
#                 splittedX = cancerousPatches[spl: splits[j+1],:]
#                 splittedX = splittedX.to(device)

#                 preds = classifier(splittedX)
#                 preds = torch.nn.functional.softmax(preds, dim=1)
#                 predictions = [*predictions, *preds.detach().cpu().numpy()]
#                 del splittedX, preds
#         else:
#             cancerousPatches = cancerousPatches.to(device)
#             preds = classifier(cancerousPatches)
#             preds = torch.nn.functional.softmax(preds, dim=1)
#             predictions = [ *preds.detach().cpu().numpy()]
#             del preds
            
#         pred = np.mean(predictions, axis=0)
# #         pred = np.max(predictions, axis=0)

# #         print(pred)
#         targetLabel = np.argmax(pred)
#         prob = pred[targetLabel]

#         label = enc.inverse_transform([targetLabel])[0]
#         result.append([fid, label])
#         del cancerousPatches


In [35]:
df = pd.DataFrame(result, columns=['image_id', "label"])
df.head()

,image_id,label
0,41,HGSC
1,99,HGSC
2,13568,HGSC
3,17637,HGSC
4,12442,HGSC


In [27]:
# df.to_csv("./submission.csv", index=False)